# 509 — Kong MCP Gateway ACL Policy (Phase 509)

**What this notebook covers:**

1. What Phase 509 adds
2. Transport vs ACL vs app logic — three distinct layers
3. Version requirements
4. Current role model from code
5. Konnect UI walkthrough
6. decK declarative config examples
7. Testing jr vs sr analyst access
8. Toggle behavior (env flags + UI backend selection)
9. Debug mode explanation
10. Troubleshooting
11. Rollback

---

## 1. What Phase 509 Adds

Phase 509 adds **gateway-enforced MCP tool access control** via Kong consumer groups.

Before Phase 509 (Phases 507–508), Kong acted as a **transport layer only**:
- Kong validated the API key (key-auth)
- Kong forwarded MCP requests to the upstream Railway server
- Tool access control was enforced **by the app** using `policy.py`

After Phase 509, when enabled, Kong **enforces which MCP tools each role can call**:
- `jr-analyst` consumers → restricted tool set
- `sr-analyst` consumers → all tools allowed
- The app surrenders tool filtering to the gateway (does NOT double-filter)

The app-side policy (`policy.py`) remains the fallback when Kong ACL is disabled.

---

## 2. Three Distinct Layers

```
┌─────────────────────────────────────────────────────────────────────┐
│  Layer 1: MCP Transport                                             │
│  Phase 507/508 — Kong sits between app and upstream MCP server.     │
│  Controls: routing, key-auth, observability, timeouts.             │
│  Always active when mcp_mode="kong" and KONG_MCP_GATEWAY_ENABLED.   │
├─────────────────────────────────────────────────────────────────────┤
│  Layer 2: Kong ACL (Phase 509)                                      │
│  ai-mcp-proxy plugin with consumer_groups.                          │
│  Controls: which MCP tools each consumer group can invoke.          │
│  Active only when KONG_MCP_ACL_POLICY_ENABLED=true AND             │
│            mcp_mode="kong" AND KONG_MCP_GATEWAY_ENABLED=true.       │
├─────────────────────────────────────────────────────────────────────┤
│  Layer 3: App policy (policy.py)                                    │
│  Always present. Active as tool filter when Kong ACL is NOT active. │
│  Controls: tab access, allowed_mcp_tools passed to Orchestrator.    │
└─────────────────────────────────────────────────────────────────────┘
```

### Enforcement Matrix

| UI Backend | `KONG_MCP_GATEWAY_ENABLED` | `KONG_MCP_ACL_POLICY_ENABLED` | Tool enforcement |
|------------|---------------------------|-------------------------------|------------------|
| Local MCP  | any                       | any                           | App policy only  |
| Remote MCP | any                       | any                           | App policy only  |
| Kong MCP   | `false`                   | any                           | Fallback → Remote MCP + app policy |
| Kong MCP   | `true`                    | `false`                       | Kong transport + app policy |
| Kong MCP   | `true`                    | `true`                        | **Kong enforces ACL** (app does NOT filter) |

**Do NOT mix both at runtime.** When Kong ACL is active, the app passes `allowed_tools=None` to the Orchestrator — Kong is the authority.

---

## 3. Version Requirements

| Component | Minimum version | Notes |
|-----------|----------------|-------|
| Kong Gateway | 3.13+ | `ai-mcp-proxy` consumer_groups ACL support |
| Kong Konnect | Serverless plan | AI Gateway Enterprise license |
| decK | 1.x | for declarative config diff/sync |
| Python app | Phase 508+ | `KongMCPToolClient` in place |

> **Note:** Phase 507 uses `ai-mcp-proxy` in `passthrough-listener` mode (Kong 3.12+).
> Phase 509 adds `consumer_groups` ACL config which requires **3.13+**.
> Verify your Konnect Serverless gateway version before enabling.

---

## 4. Current Role Model (from code)

The role model lives in `src/app/policy.py`. Phase 509 maps app roles → Kong consumer groups.

In [1]:
import sys
sys.path.insert(0, "..")

from src.app.policy import (
    ROLE_JR_RISK_ANALYST,
    ROLE_SR_RISK_ANALYST,
    JR_ALLOWED_MCP_TOOLS,
    ALL_MCP_TOOLS,
    _JR_DENIED_TOOLS,
    KONG_CONSUMER_GROUP_MAP,
    get_kong_consumer_group,
)

print("=== Role → Kong Consumer Group mapping ===")
for role, group in KONG_CONSUMER_GROUP_MAP.items():
    print(f"  {role!r:30s} → {group!r}")

print()
print(f"=== Jr analyst ({ROLE_JR_RISK_ANALYST}) ===")
print(f"  Allowed tools ({len(JR_ALLOWED_MCP_TOOLS)}):")
for t in sorted(JR_ALLOWED_MCP_TOOLS):
    print(f"    ✓ {t}")
print(f"  Denied tools ({len(_JR_DENIED_TOOLS)}):")
for t in sorted(_JR_DENIED_TOOLS):
    print(f"    ✗ {t}")

print()
print(f"=== Sr analyst ({ROLE_SR_RISK_ANALYST}) ===")
print(f"  Allowed tools ({len(ALL_MCP_TOOLS)}): all tools")

=== Role → Kong Consumer Group mapping ===
  'jr_risk_analyst'              → 'jr-analyst'
  'sr_risk_analyst'              → 'sr-analyst'

=== Jr analyst (jr_risk_analyst) ===
  Allowed tools (13):
    ✓ company_profile
    ✓ control_signal_check
    ✓ entity_lookup
    ✓ evaluate_stop_conditions
    ✓ expand_ownership
    ✓ find_traces_by_entity
    ✓ list_recent_traces
    ✓ ownership_complexity_check
    ✓ resolve_entity
    ✓ retrieve_trace
    ✓ shared_address_check
    ✓ sic_context
    ✓ validate_plan
  Denied tools (3):
    ✗ address_risk_check
    ✗ industry_context_check
    ✗ summarize_risk_for_company

=== Sr analyst (sr_risk_analyst) ===
  Allowed tools (15): all tools


In [2]:
# Check the enforcement helper
from src.config import KongMCPGatewaySettings, kong_acl_enforcement_active

def _make_settings(enabled: bool, acl_enabled: bool) -> KongMCPGatewaySettings:
    return KongMCPGatewaySettings(
        enabled=enabled,
        proxy_url="https://example.kong.tech",
        route_path="/mcp",
        api_key="test-key",
        upstream_url="https://upstream/mcp",
        acl_policy_enabled=acl_enabled,
        jr_api_key="",
        sr_api_key="",
    )

cases = [
    ("local",  True,  True,  False),
    ("remote", True,  True,  False),
    ("kong",   False, True,  False),
    ("kong",   True,  False, False),
    ("kong",   True,  True,  True),   # ← only this one activates
]

print("mcp_mode | gw_enabled | acl_enabled | kong_acl_active")
print("-" * 56)
for mcp_mode, gw_enabled, acl_enabled, expected in cases:
    s = _make_settings(gw_enabled, acl_enabled)
    result = kong_acl_enforcement_active(mcp_mode, s)
    mark = "✓" if result == expected else "✗ MISMATCH"
    print(f"{mcp_mode:8s} | {str(gw_enabled):10s} | {str(acl_enabled):11s} | {str(result):5s}  {mark}")

mcp_mode | gw_enabled | acl_enabled | kong_acl_active
--------------------------------------------------------
local    | True       | True        | False  ✓
remote   | True       | True        | False  ✓
kong     | False      | True        | False  ✓
kong     | True       | False       | False  ✓
kong     | True       | True        | True   ✓


---

## 5. Konnect UI Walkthrough

### Step 1: Create Consumer Groups

1. Open **Konnect → Gateway Manager → entity-risk-ai control plane**
2. Navigate to **Consumers → Consumer Groups**
3. Click **+ New Consumer Group**
   - Name: `jr-analyst` → Save
4. Click **+ New Consumer Group**
   - Name: `sr-analyst` → Save

### Step 2: Create Consumers and Assign Groups

**Jr analyst consumer:**
1. Navigate to **Consumers → + New Consumer**
   - Username: `jr-analyst-app` → Save
2. Open `jr-analyst-app` → **Credentials → Key Auth → + New Credential**
   - Key: `<value of KONG_MCP_ACL_JR_API_KEY>` → Save
3. Open `jr-analyst-app` → **Consumer Groups → + Add to Group**
   - Select `jr-analyst` → Save

**Sr analyst consumer:**
1. Navigate to **Consumers → + New Consumer**
   - Username: `sr-analyst-app` → Save
2. Open `sr-analyst-app` → **Credentials → Key Auth → + New Credential**
   - Key: `<value of KONG_MCP_ACL_SR_API_KEY>` → Save
3. Open `sr-analyst-app` → **Consumer Groups → + Add to Group**
   - Select `sr-analyst` → Save

### Step 3: Configure ai-mcp-proxy ACL on mcp-route

1. Navigate to **Routes → mcp-route → Plugins**
2. Find **ai-mcp-proxy** → Edit

Set top-level fields:
- `consumer_identifier`: `username`
- `include_consumer_groups`: `true`
- `default_acl`: `[]`  ← empty = allow all tools by default

**ACL model: per-tool deny lists**

Rather than maintaining a per-group allowlist, configure a `deny` list on each
tool you want to restrict. Tools with `acl: null` are unrestricted.

Example — deny `resolve_entity` for `jr-analyst`:
```yaml
tools:
  - name: resolve_entity
    acl:
      allow: null
      deny:
        - jr-analyst
  # all other tools: acl: null (no restriction)
```

With `default_acl: []`, only tools that explicitly list a group in their `deny`
field will be blocked for that group. Everything else passes through.

### Step 4: Enable ACL in App

```bash
# In .env:
KONG_MCP_GATEWAY_ENABLED=true
KONG_MCP_ACL_POLICY_ENABLED=true
KONG_MCP_ACL_JR_API_KEY=<key created for jr-analyst-app consumer>
KONG_MCP_ACL_SR_API_KEY=<key created for sr-analyst-app consumer>
```

The app selects the correct key automatically based on the signed-in user's role
via `KongMCPGatewaySettings.resolve_api_key(role)` in `factory.py`.

---

## 6. decK Config Examples

Full declarative configs are in:
- `kong/consumer_groups.yaml` — consumers, groups, credentials
- `kong/acl_policy.yaml` — ai-mcp-proxy plugin with ACL rules

**Always use `diff` before `sync`:**

```bash
# Diff consumer groups
deck gateway diff \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong/consumer_groups.yaml

# Diff ACL policy
deck gateway diff \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong/acl_policy.yaml
```

> ⚠️ **Do NOT run `deck gateway sync` without reviewing the diff first.** These
> files represent partial configs — syncing without the full live state may wipe
> existing routes and plugins not represented here.

**To dump the full live config:**
```bash
deck gateway dump \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  -o kong/declarative/current-live-state.yaml
```

---

## 7. Testing Jr vs Sr Analyst Access

Use the example scripts in `kong/examples/`:

```bash
# Jr analyst — expects resolve_entity DENIED, entity_lookup ALLOWED
bash kong/examples/phase509_jr_analyst_test.sh

# Sr analyst — expects all tools ALLOWED
bash kong/examples/phase509_sr_analyst_test.sh
```

Or use the notebook cells below for a Python-based smoke test.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

KONG_PROXY_URL = os.getenv("KONG_PROXY_URL", "")
KONG_MCP_GATEWAY_ROUTE_PATH = os.getenv("KONG_MCP_GATEWAY_ROUTE_PATH", "/mcp")
JR_API_KEY = os.getenv("KONG_MCP_ACL_JR_API_KEY", "")
SR_API_KEY = os.getenv("KONG_MCP_ACL_SR_API_KEY", "")
ENABLE_LIVE = os.getenv("ENABLE_LIVE_KONG_NOTEBOOK_TESTS", "").strip().lower() == "true"

MCP_ENDPOINT = KONG_PROXY_URL.rstrip("/") + KONG_MCP_GATEWAY_ROUTE_PATH if KONG_PROXY_URL else ""

print("Live test gate:", ENABLE_LIVE)
print("MCP endpoint:", MCP_ENDPOINT or "(not set)")
print("Jr API key:", (JR_API_KEY[:4] + "…***") if JR_API_KEY else "(not set)")
print("Sr API key:", (SR_API_KEY[:4] + "…***") if SR_API_KEY else "(not set)")

Live test gate: True
MCP endpoint: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp
Jr API key: abc1…***
Sr API key: abc1…***


In [4]:
# Live ACL smoke test — all 15 MCP tools, both roles.
#
# ACL model in use (per-tool deny lists, default_acl: []):
#   resolve_entity: deny [jr-analyst]   → jr DENIED, sr ALLOWED
#   all other tools: acl null           → everyone ALLOWED
#
# Result codes:
#   DENIED  — Kong ACL blocked the call (Forbidden / ACL error)
#   ALLOWED — tool reached the upstream and ran (data/param errors are fine)
#   ERROR   — unexpected transport or protocol error (not ACL)

if not ENABLE_LIVE:
    print("⏭ Skipping live test (set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run)")
elif not MCP_ENDPOINT:
    print("⏭ Skipping: KONG_PROXY_URL not set")
elif not JR_API_KEY or not SR_API_KEY:
    print("⏭ Skipping: KONG_MCP_ACL_JR_API_KEY or KONG_MCP_ACL_SR_API_KEY not set")
else:
    from mcp import ClientSession
    from mcp.client.streamable_http import streamablehttp_client

    # All 15 MCP tools with minimal valid parameters.
    # ACL is checked before tool execution, so dummy values are fine for ACL probing.
    TOOLS = [
        ("resolve_entity",             {"name": "TEST LTD"}),
        ("validate_plan",              {"steps": []}),
        ("evaluate_stop_conditions",   {"findings": {}}),
        ("entity_lookup",              {"name": "TEST LTD"}),
        ("company_profile",            {"company_number": "00000000"}),
        ("expand_ownership",           {"company_name": "TEST LTD"}),
        ("shared_address_check",       {"company_name": "TEST LTD"}),
        ("sic_context",                {"company_name": "TEST LTD"}),
        ("ownership_complexity_check", {"company_name": "TEST LTD"}),
        ("control_signal_check",       {"company_name": "TEST LTD"}),
        ("retrieve_trace",             {"trace_id": "test-trace-id"}),
        ("find_traces_by_entity",      {"entity_name": "TEST LTD"}),
        ("list_recent_traces",         {}),
        ("address_risk_check",         {"company_name": "TEST LTD"}),
        ("industry_context_check",     {"company_name": "TEST LTD"}),
    ]

    def _unwrap(exc: BaseException) -> BaseException:
        if hasattr(exc, "exceptions") and exc.exceptions:
            return _unwrap(exc.exceptions[0])
        return exc

    def _is_acl_error(exc: BaseException) -> bool:
        msg = str(exc).lower()
        return "forbidden" in msg or "acl" in msg or "not allowed" in msg

    async def probe_role(api_key: str, role_label: str) -> None:
        headers = {"X-Kong-API-Key": api_key, "Accept": "text/event-stream"}
        async with streamablehttp_client(MCP_ENDPOINT, headers=headers) as (r, w, _):
            async with ClientSession(r, w) as session:
                await session.initialize()
                for tool_name, params in TOOLS:
                    try:
                        result = await session.call_tool(tool_name, params)
                        # Tool reached upstream — ACL allowed it (execution errors are fine)
                        status = "ALLOWED"
                    except Exception as exc:
                        real = _unwrap(exc)
                        if _is_acl_error(real):
                            status = f"DENIED  ({type(real).__name__}: {str(real)[:60]})"
                        else:
                            status = f"ERROR   ({type(real).__name__}: {str(real)[:60]})"
                    print(f"  [{role_label}] {tool_name:<30s} {status}")

    print("=== Jr analyst (jr-analyst consumer group) ===")
    await probe_role(JR_API_KEY, "jr")

    print()
    print("=== Sr analyst (sr-analyst consumer group) ===")
    await probe_role(SR_API_KEY, "sr")

=== Jr analyst (jr-analyst consumer group) ===
  [jr] resolve_entity                 ALLOWED
  [jr] validate_plan                  ALLOWED
  [jr] evaluate_stop_conditions       ALLOWED
  [jr] entity_lookup                  ALLOWED
  [jr] company_profile                ALLOWED
  [jr] expand_ownership               ALLOWED
  [jr] shared_address_check           ALLOWED
  [jr] sic_context                    ALLOWED
  [jr] ownership_complexity_check     ALLOWED
  [jr] control_signal_check           ALLOWED
  [jr] retrieve_trace                 ALLOWED
  [jr] find_traces_by_entity          ALLOWED
  [jr] list_recent_traces             ALLOWED
  [jr] address_risk_check             DENIED  (McpError: Forbidden by tool ACL)
  [jr] industry_context_check         DENIED  (McpError: Forbidden by tool ACL)

=== Sr analyst (sr-analyst consumer group) ===
  [sr] resolve_entity                 ALLOWED
  [sr] validate_plan                  ALLOWED
  [sr] evaluate_stop_conditions       ALLOWED
  [sr] en

---

## 8. Toggle Behavior

### Environment flags

| Flag | Effect |
|------|--------|
| `KONG_MCP_GATEWAY_ENABLED=false` | Kong transport disabled; app uses local or remote MCP |
| `KONG_MCP_GATEWAY_ENABLED=true`, `KONG_MCP_ACL_POLICY_ENABLED=false` | Kong transport active, app-side policy enforces tools |
| `KONG_MCP_GATEWAY_ENABLED=true`, `KONG_MCP_ACL_POLICY_ENABLED=true` | Kong ACL active (when UI backend = "kong") |

### UI backend selection

The Streamlit sidebar "Tool backend" radio determines which transport is used:
- **Local (in-process)** — `MCPToolClient`, app policy always active
- **Remote MCP Server** — `RemoteMCPToolClient`, app policy always active
- **Kong MCP Gateway** — `KongMCPToolClient`, Kong ACL active if both flags are true

Kong ACL is **never** active when the UI backend is Local or Remote — regardless of env flags.

### Code path (layout.py)

```python
# Phase 509 enforcement switch (simplified)
_kong_mcp_settings = get_kong_mcp_gateway_settings()
_kong_acl_active = kong_acl_enforcement_active(mcp_mode, _kong_mcp_settings)

if _kong_acl_active:
    _allowed = None          # Kong enforces — app does NOT filter
    log "Kong MCP ACL policy ACTIVE — role=... endpoint=..."
else:
    _allowed = policy.allowed_mcp_tools   # app-side filter
```

---

## 9. Troubleshooting

### Problem: Kong ACL not triggering

Check all three conditions:
```python
from src.config import get_kong_mcp_gateway_settings, kong_acl_enforcement_active
s = get_kong_mcp_gateway_settings()
print(s.masked())
# mcp_mode must be "kong" at runtime (check Streamlit sidebar selection)
print("ACL active:", kong_acl_enforcement_active("kong", s))
```

### Problem: All users get the same ACL (shared key limitation)

**Root cause:** A single `KONG_MCP_GATEWAY_API_KEY` maps all users to the same
Kong consumer — Kong cannot distinguish jr from sr analysts.

**Phase 509 solution (implemented):** The app resolves a per-role key at component
creation time via `KongMCPGatewaySettings.resolve_api_key(role)`:
- `KONG_MCP_ACL_JR_API_KEY` → `jr-analyst-app` consumer → `jr-analyst` group
- `KONG_MCP_ACL_SR_API_KEY` → `sr-analyst-app` consumer → `sr-analyst` group

Set both keys in `.env`. If a role-specific key is absent, `resolve_api_key()`
falls back to the shared `KONG_MCP_GATEWAY_API_KEY` — no crash, degraded isolation.

**Future path (identity propagation):**
1. App mints a short-lived JWT containing the role claim after login
2. JWT is sent as `Authorization: Bearer <token>` to Kong
3. Kong `jwt` plugin validates the claim → maps to consumer → consumer group
4. ACL enforced by consumer group membership
5. `auth_provider` in `AuthenticatedUser` changes from `"mock"` to `"kong"`

### Problem: Kong returns 403 for a tool the role should have

1. Check `kong/acl_policy.yaml` allowed_tools list matches `src/app/policy.py`
2. Check the consumer is in the correct group (Konnect UI → Consumers)
3. Check the API key used maps to the correct consumer
4. Check Kong Gateway version ≥ 3.13

### Problem: app-side policy also filtering when Kong ACL active

Ensure `KONG_MCP_ACL_POLICY_ENABLED=true` and `KONG_MCP_GATEWAY_ENABLED=true`
and that the UI backend radio shows "Kong MCP Gateway".
When all three conditions are true, the app passes `allowed_tools=None` to
the Orchestrator — app-side filtering is disabled.

In [5]:
# Quick settings dump for debugging
from src.config import get_kong_mcp_gateway_settings, kong_acl_enforcement_active
import pprint

s = get_kong_mcp_gateway_settings()
print("=== KongMCPGatewaySettings ===")
pprint.pprint(s.masked())

print()
for mode in ("local", "remote", "kong"):
    active = kong_acl_enforcement_active(mode, s)
    print(f"  mcp_mode={mode!r:8s}  kong_acl_active={active}")

=== KongMCPGatewaySettings ===
{'acl_policy_enabled': True,
 'api_key': 'KsxhEYlr…***',
 'enabled': True,
 'gateway_url': 'https://kong-948ef205bdausa0ym.kongcloud.dev/mcp',
 'jr_api_key': 'abc123jr…***',
 'proxy_url': 'https://kong-948ef205bdausa0ym.kongcloud.dev',
 'route_path': '/mcp',
 'sr_api_key': 'abc123sr…***',
 'upstream_url': 'https://entity-risk-ai-production.up.railway.app'}

  mcp_mode='local'   kong_acl_active=False
  mcp_mode='remote'  kong_acl_active=False
  mcp_mode='kong'    kong_acl_active=True


---

## 10. Debug Mode Explanation

When `KONG_MCP_ACL_POLICY_ENABLED=false` (or any non-kong backend is selected),
the app falls back to **app-based debug mode**:

- `policy.py` `allowed_mcp_tools` is passed to the Orchestrator
- The Orchestrator skips steps for disallowed tools
- Useful for local development without a live Kong instance
- Useful for testing policy changes before deploying to Kong

This means you can develop and test the full ACL logic locally using only
the app — no Kong required. When satisfied, deploy the consumer group config
to Konnect and enable both flags.

---

## 11. Rollback

### Immediate rollback (fastest)
```bash
# In .env:
KONG_MCP_ACL_POLICY_ENABLED=false
```
Restart the Streamlit app. App-side policy becomes active immediately.
No Kong config changes required.

### Partial rollback (remove ACL from Kong, keep transport)
1. Konnect UI → Routes → mcp-route → Plugins → ai-mcp-proxy → Edit
2. Remove `consumer_groups` entries
3. Remove `consumer_identifier` and `include_consumer_groups` fields
4. Save
5. Set `KONG_MCP_ACL_POLICY_ENABLED=false` in app `.env`

### Full rollback (revert to Phase 507)
1. Run the Phase 507 reference config diff:
   ```bash
   deck gateway diff ... kong/declarative/mcp-gateway-reference.yaml
   ```
2. Review diff carefully (consumer groups and ACL rules will appear as deletions)
3. Apply if the diff looks correct:
   ```bash
   deck gateway sync ... kong/declarative/mcp-gateway-reference.yaml
   ```
4. Set `KONG_MCP_ACL_POLICY_ENABLED=false` in app `.env`
5. Delete `jr-analyst-app` and `sr-analyst-app` consumers from Konnect UI

---

## Summary

Phase 509 adds gateway-enforced MCP tool ACL on top of the Phase 507/508 transport:

- **New env flag:** `KONG_MCP_ACL_POLICY_ENABLED`
- **New config:** `KongMCPGatewaySettings.acl_policy_enabled` + `kong_acl_enforcement_active()`
- **New policy helpers:** `KONG_CONSUMER_GROUP_MAP`, `get_kong_consumer_group()`
- **New Kong assets:** `kong/consumer_groups.yaml`, `kong/acl_policy.yaml`, `kong/examples/`
- **App behavior:** when ACL active → `allowed_tools=None` → Kong enforces; otherwise → app policy
- **Limitation:** shared API key cannot differentiate roles; per-role keys required
- **No regressions:** Local MCP, Remote MCP, and Kong-transport-only modes unchanged